<a href="https://colab.research.google.com/github/Ian072/bachealert/blob/main/ml/entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile, os, glob

zip_path = (glob.glob('/content/drive/MyDrive/bachealert*.zip') +
            glob.glob('/content/drive/MyDrive/archive.zip'))[0]
print("Usando:", zip_path)

with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/data')

# Busca las dos carpetas de fotos, estén donde estén dentro del zip
carpetas = [d for d in glob.glob('/content/data/**/', recursive=True)
            if os.path.basename(d.rstrip('/')).lower() in ('normal', 'potholes')]
DATA_DIR = os.path.dirname(carpetas[0].rstrip('/'))
print("Carpeta de datos:", DATA_DIR)
for c in sorted(os.listdir(DATA_DIR)):
    print(c, "->", len(os.listdir(os.path.join(DATA_DIR, c))), "fotos")

Usando: /content/drive/MyDrive/bachealert.zip
Carpeta de datos: /content/data
normal -> 352 fotos
potholes -> 329 fotos


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
print("Clases:", dataset.classes)   # ['normal', 'potholes']

n = len(dataset)
n_train = int(0.7 * n); n_val = int(0.15 * n); n_test = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                         generator=torch.Generator().manual_seed(42))

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32)
test_dl  = DataLoader(test_ds, batch_size=32)
print(f"Entrenamiento: {n_train}, Validación: {n_val}, Prueba: {n_test}")

Clases: ['normal', 'potholes']
Entrenamiento: 476, Validación: 102, Prueba: 103


In [ ]:
import torch.nn as nn
from torchvision import models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Usando:", device)

model = models.mobilenet_v2(weights='DEFAULT')   # ya sabe "ver"
for p in model.features.parameters():
    p.requires_grad = False                       # congelamos lo que ya sabe
model.classifier[1] = nn.Linear(1280, 2)          # solo entrenamos la última capa: 2 clases
model = model.to(device)

Usando: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 108MB/s] 


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

def evaluar(dl):
    model.eval(); correctos = 0; total = 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correctos += (pred == y).sum().item(); total += len(y)
    return correctos / total

for epoca in range(5):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    print(f"Época {epoca+1}: exactitud en validación = {evaluar(val_dl):.1%}")

Época 1: exactitud en validación = 96.1%
Época 2: exactitud en validación = 100.0%
Época 3: exactitud en validación = 100.0%
Época 4: exactitud en validación = 100.0%
Época 5: exactitud en validación = 100.0%


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval(); y_real = []; y_pred = []
with torch.no_grad():
    for x, y in test_dl:
        y_pred += model(x.to(device)).argmax(1).cpu().tolist()
        y_real += y.tolist()

print(f"Exactitud: {accuracy_score(y_real, y_pred):.1%}")
print(f"Precisión: {precision_score(y_real, y_pred):.1%}  (de lo que dijo 'bache', cuánto sí era)")
print(f"Recall:    {recall_score(y_real, y_pred):.1%}  (de los baches reales, cuántos detectó)")

Exactitud: 95.1%
Precisión: 96.2%  (de lo que dijo 'bache', cuánto sí era)
Recall:    94.4%  (de los baches reales, cuántos detectó)


In [ ]:
!pip install -q onnx onnxscript

In [ ]:
os.makedirs('/content/drive/MyDrive/bachealert/modelo', exist_ok=True)
torch.save(model.state_dict(), '/content/drive/MyDrive/bachealert/modelo/bachealert.pt')

model.cpu().eval()
ejemplo = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, ejemplo, '/content/drive/MyDrive/bachealert/modelo/bachealert.onnx',
                  input_names=['imagen'], output_names=['salida'], opset_version=17)
model.to(device)
print("Guardado en Drive: bachealert.pt y bachealert.onnx")

W1007 16:10:35.525000 2336 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:55: 

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Guardado en Drive: bachealert.pt y bachealert.onnx


In [ ]:
from google.colab import files
from PIL import Image

subida = files.upload()   # elige una foto de tu computadora
ruta = list(subida.keys())[0]

img = transform(Image.open(ruta).convert('RGB')).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(img), dim=1)[0]
print(f"Bache: {probs[1]:.1%}   No bache: {probs[0]:.1%}")
print("Veredicto:", "ES BACHE" if probs[1] > 0.5 else "NO ES BACHE")

Saving Bache 2.jpg to Bache 2 (1).jpg
Bache: 49.0%   No bache: 51.0%
Veredicto: NO ES BACHE
